In [7]:
import pandas as pd
import numpy as np

# 1. Chargement du Dataset A brut
# Remplace 'dataset_A.csv' par le nom exact de ton fichier dans data/raw/ s'il est différent
df = pd.read_csv('../data/raw/traffic_weather_full2020.csv', low_memory=False)
print("=== ÉTAPE 1 TERMINÉE ===")

# 2. Aperçu des 5 premières lignes
print("=== 5 PREMIÈRES LIGNES ===")
display(df.head())

# 3. Colonnes et types de données enregistrés par Pandas
print("\n=== TYPES DE DONNÉES ET COLONNES ===")
print(df.info())

# 4. Comptage des valeurs manquantes (cases vides)
print("\n=== VALEURS MANQUANTES ===")
print(df.isnull().sum())


=== ÉTAPE 1 TERMINÉE ===
=== 5 PREMIÈRES LIGNES ===


,5 Minutes,Flow,(mph),week_day,holidays,Hour,Day,Time,Temperature,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Precip.,Condition
0,2/1/2020 0:00,338,66.0,6,1,2/1/2020 0:00,2/1/2020,12:00,55 F,29 F,37.0,VAR,3,0,29.39,0.0,Fair
1,2/1/2020 0:05,360,65.5,6,1,2/1/2020 0:05,2/1/2020,12:00,55 F,29 F,37.0,VAR,3,0,29.39,0.0,Fair
2,2/1/2020 0:10,347,65.2,6,1,2/1/2020 0:10,2/1/2020,12:00,55 F,29 F,37.0,VAR,3,0,29.39,0.0,Fair
3,2/1/2020 0:15,337,65.4,6,1,2/1/2020 0:15,2/1/2020,12:00,55 F,29 F,37.0,VAR,3,0,29.39,0.0,Fair
4,2/1/2020 0:20,302,66.3,6,1,2/1/2020 0:20,2/1/2020,12:00,55 F,29 F,37.0,VAR,3,0,29.39,0.0,Fair



=== TYPES DE DONNÉES ET COLONNES ===
<class 'pandas.DataFrame'>
RangeIndex: 34823 entries, 0 to 34822
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   5 Minutes    34823 non-null  str    
 1   Flow         34823 non-null  int64  
 2   (mph)        34823 non-null  float64
 3   week_day     34823 non-null  int64  
 4   holidays     34823 non-null  int64  
 5   Hour         34823 non-null  str    
 6   Day          34823 non-null  str    
 7   Time         34823 non-null  str    
 8   Temperature  34823 non-null  str    
 9   Dew Point    34823 non-null  str    
 10  Humidity     34823 non-null  float64
 11  Wind         34655 non-null  str    
 12  Wind Speed   34823 non-null  str    
 13  Wind Gust    34823 non-null  str    
 14  Pressure     34823 non-null  float64
 15  Precip.      34823 non-null  float64
 16  Condition    34799 non-null  str    
dtypes: float64(4), int64(3), str(10)
memory usage: 6.6 MB
None

=

In [8]:
# 1. Conversion de la colonne temporelle en format datetime
df['datetime'] = pd.to_datetime(df['5 Minutes'])

# 2. Tri chronologique et réinitialisation de l'index
df = df.sort_values('datetime').reset_index(drop=True)

# 3. Traitement des valeurs manquantes par propagation (ffill)
df['Condition'] = df['Condition'].ffill()

# 4. Vérification après l'Étape 2
print("=== ÉTAPE 2 TERMINÉE ===")
print("Nombre total de valeurs manquantes restantes :", df.isnull().sum().sum())
print("Plage temporelle du dataset : de", df['datetime'].min(), "à", df['datetime'].max())

=== ÉTAPE 2 TERMINÉE ===
Nombre total de valeurs manquantes restantes : 168
Plage temporelle du dataset : de 2020-02-01 00:00:00 à 2020-05-31 23:55:00


In [9]:
# 1. Définition des 9 colonnes retenues
selected_columns = [
    'datetime', 'Flow', '(mph)', 'week_day', 'holidays',
    'Temperature', 'Wind Speed', 'Precip.', 'Condition'
]

# 2. Application du filtrage
df_clean = df[selected_columns].copy()

# 3. Affichage de la structure intermédiaire
print("=== ÉTAPE 3 TERMINÉE ===")
print(f"Dimension après filtrage : {df_clean.shape[0]} lignes et {df_clean.shape[1]} colonnes")
display(df_clean.head())

=== ÉTAPE 3 TERMINÉE ===
Dimension après filtrage : 34823 lignes et 9 colonnes


,datetime,Flow,(mph),week_day,holidays,Temperature,Wind Speed,Precip.,Condition
0,2020-02-01 00:00:00,338,66.0,6,1,55 F,3,0.0,Fair
1,2020-02-01 00:05:00,360,65.5,6,1,55 F,3,0.0,Fair
2,2020-02-01 00:10:00,347,65.2,6,1,55 F,3,0.0,Fair
3,2020-02-01 00:15:00,337,65.4,6,1,55 F,3,0.0,Fair
4,2020-02-01 00:20:00,302,66.3,6,1,55 F,3,0.0,Fair


In [10]:
# 1. Copie de travail
df_si = df_clean.copy()

# 2. Conversion préalable en type numérique (extraction des chiffres si du texte traîne)
df_si['(mph)'] = pd.to_numeric(df_si['(mph)'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df_si['Temperature'] = pd.to_numeric(df_si['Temperature'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df_si['Wind Speed'] = pd.to_numeric(df_si['Wind Speed'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df_si['Precip.'] = pd.to_numeric(df_si['Precip.'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')

# 3. Conversions mathématiques vers le Système International (SI)
df_si['speed_kmh'] = (df_si['(mph)'] * 1.60934).round(2)
df_si['temp_celsius'] = ((df_si['Temperature'] - 32) * 5 / 9).round(2)
df_si['wind_speed_kmh'] = (df_si['Wind Speed'] * 1.60934).round(2)
df_si['precip_mm'] = (df_si['Precip.'] * 25.4).round(2)

# 4. Renommage et réorganisation des colonnes
df_si = df_si.rename(columns={
    'Flow': 'traffic_flow',
    'Condition': 'weather_condition'
})

final_columns = [
    'datetime', 'traffic_flow', 'speed_kmh', 'week_day', 'holidays',
    'temp_celsius', 'wind_speed_kmh', 'precip_mm', 'weather_condition'
]
df_si = df_si[final_columns]

# 5. Vérification de l'Étape 4
print("=== ÉTAPE 4 TERMINÉE ===")
print("Nouvelles colonnes et unités SI :")
display(df_si.head())

=== ÉTAPE 4 TERMINÉE ===
Nouvelles colonnes et unités SI :


,datetime,traffic_flow,speed_kmh,week_day,holidays,temp_celsius,wind_speed_kmh,precip_mm,weather_condition
0,2020-02-01 00:00:00,338,106.22,6,1,12.78,4.83,0.0,Fair
1,2020-02-01 00:05:00,360,105.41,6,1,12.78,4.83,0.0,Fair
2,2020-02-01 00:10:00,347,104.93,6,1,12.78,4.83,0.0,Fair
3,2020-02-01 00:15:00,337,105.25,6,1,12.78,4.83,0.0,Fair
4,2020-02-01 00:20:00,302,106.70,6,1,12.78,4.83,0.0,Fair


In [11]:
# 1. Copie du DataFrame SI
df_fe = df_si.copy()

# 2. Extractions temporelles depuis datetime
df_fe['hour'] = df_fe['datetime'].dt.hour
df_fe['minute'] = df_fe['datetime'].dt.minute

# CORRECTION IS_WEEKEND : Pandas dayofweek (Lundi=0, ..., Samedi=5, Dimanche=6)
df_fe['is_weekend'] = (df_fe['datetime'].dt.day_of_week >= 5).astype(int)

# 3. Création des Lag Features sur le trafic
df_fe['traffic_flow_lag_1'] = df_fe['traffic_flow'].shift(1)
df_fe['traffic_flow_lag_3'] = df_fe['traffic_flow'].shift(3)

# 4. Suppression des 3 premières lignes n'ayant pas d'historique (NaN)
df_fe = df_fe.dropna().reset_index(drop=True)

# 5. Vérification
print("=== ÉTAPE 5 TERMINÉE ===")
print(f"Dimensions finales : {df_fe.shape[0]} lignes et {df_fe.shape[1]} colonnes")
display(df_fe[['datetime', 'traffic_flow', 'traffic_flow_lag_1', 'traffic_flow_lag_3', 'hour', 'is_weekend']].head())

=== ÉTAPE 5 TERMINÉE ===
Dimensions finales : 34820 lignes et 14 colonnes


,datetime,traffic_flow,traffic_flow_lag_1,traffic_flow_lag_3,hour,is_weekend
0,2020-02-01 00:15:00,337,347.0,338.0,0,1
1,2020-02-01 00:20:00,302,337.0,360.0,0,1
2,2020-02-01 00:25:00,282,302.0,347.0,0,1
3,2020-02-01 00:30:00,302,282.0,337.0,0,1
4,2020-02-01 00:35:00,277,302.0,302.0,0,1


In [12]:
import os

# 1. Création du dossier data/processed s'il n'existe pas
os.makedirs('../data/processed', exist_ok=True)

# 2. Exporter le dataset nettoyé complet
df_fe.to_csv('../data/processed/dataset_A_cleaned.csv', index=False)

# 3. Calcul des index de découpage (70% / 15% / 15%)
n = len(df_fe)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_fe.iloc[:train_end]
val_df = df_fe.iloc[train_end:val_end]
test_df = df_fe.iloc[val_end:]

# 4. Exportation des 3 jeux de données
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

# 5. Confirmation
print("=== ÉTAPE 6 TERMINÉE & FICHIERS EXPORTÉS ===")
print(f"Jeu complet : {len(df_fe)} lignes -> saved to data/processed/dataset_A_cleaned.csv")
print(f"Train (70%) : {len(train_df)} lignes -> saved to data/processed/train.csv")
print(f"Val   (15%) : {len(val_df)} lignes -> saved to data/processed/val.csv")
print(f"Test  (15%) : {len(test_df)} lignes -> saved to data/processed/test.csv")

=== ÉTAPE 6 TERMINÉE & FICHIERS EXPORTÉS ===
Jeu complet : 34820 lignes -> saved to data/processed/dataset_A_cleaned.csv
Train (70%) : 24374 lignes -> saved to data/processed/train.csv
Val   (15%) : 5223 lignes -> saved to data/processed/val.csv
Test  (15%) : 5223 lignes -> saved to data/processed/test.csv
